In [11]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('wnba_shots_2025.csv')

# Returns the exact totals for 1s (Made) and 0s (Missed)
flag_counts = df['SHOT_MADE_FLAG'].value_counts()

print(flag_counts)

SHOT_MADE_FLAG
0    21609
1    16926
Name: count, dtype: int64


In [4]:

# 1. Group periods into First 2 Periods (1–2) vs. Last 2 / OT (3+)
df['PERIOD_GROUP'] = df['PERIOD'].apply(
    lambda x: 'First 2 Periods (P1-P2)' if x <= 2 else 'Last 2 / OT (P3+)'
)

# 2. Build the truth table (cross-tabulation)
truth_table = pd.crosstab(
    index=df['PERIOD_GROUP'],
    columns=df['SHOT_MADE_FLAG'],
    margins=True,
    margins_name='Total'
).rename(columns={0: 'Missed (0)', 1: 'Made (1)'})

print(truth_table)

SHOT_MADE_FLAG           Missed (0)  Made (1)  Total
PERIOD_GROUP                                        
First 2 Periods (P1-P2)       11008      8725  19733
Last 2 / OT (P3+)             10601      8201  18802
Total                         21609     16926  38535


In [12]:
df['IS_MOVING_SHOT'] = df['ACTION_TYPE'].str.contains(
    'Pullup|Step Back|Fadeaway|Floating|Driving|Running|Cut|Cutting', 
    regex=True, case=False
)

df['MECHANIC_LABEL'] = df['IS_MOVING_SHOT'].astype(int).map({
    0: 'Set/Stationary (0)', 
    1: 'Moving/Dynamic (1)'
})

In [13]:
trio_truth_table = pd.crosstab(
    index=[df['PERIOD_GROUP'], df['MECHANIC_LABEL']], 
    columns=df['SHOT_MADE_FLAG'],
    margins=True,
    margins_name='Total'
).rename(columns={0: 'Missed (0)', 1: 'Made (1)'})

print("--- 3-WAY TRUTH TABLE (COUNTS) ---")
print(trio_truth_table)

--- 3-WAY TRUTH TABLE (COUNTS) ---
SHOT_MADE_FLAG                              Missed (0)  Made (1)  Total
PERIOD_GROUP            MECHANIC_LABEL                                 
First 2 Periods (P1-P2) Moving/Dynamic (1)        5923      5157  11080
                        Set/Stationary (0)        5085      3568   8653
Last 2 / OT (P3+)       Moving/Dynamic (1)        5589      4951  10540
                        Set/Stationary (0)        5012      3250   8262
Total                                            21609     16926  38535
